In [ ]:
import random
import re
import os
import asyncio
import pandas as pd
import shutil
import json
from itertools import chain
import pandas as pd
import zipfile

# Download FixEval Dataset from https://docs.google.com/uc?export=download&id=1rjjYW8SB8f5Hr34ig84OKpNYOzdt03Ar
# Docs: https://github.com/mahimanzum/FixEval

In [ ]:
# 1) Read all JSON files from the ZIP archive
data = []
with zipfile.ZipFile("./data/FixEval_dataset.zip", "r") as z:
    for file_index in range(0,115): # indexes of files
        print(file_index, end=', ')
        filename = f"FixEval_dataset/{file_index}.json"
        try:
            with z.open(f"{filename}") as f:
                # Read as bytes → decode → json.loads
                file_data = json.loads(f.read().decode("utf-8"))
                data += file_data
        except Exception as e:
            print(f"Could not read {filename}: {e}")

# 2) Flatten the outer list-of-lists
records = list(chain.from_iterable(data))   # -> [{...}, {...}, {...}, ...]

# 3) Normalize into a DataFrame
df = pd.json_normalize(records)

# Optional: ensure text columns are strings (helpful for multiline code)
text_cols = ["source", "lang", "problem_id", "code_tokens", "submission_id", "verdict"]
for c in text_cols:
    if c in df.columns:
        df[c] = df[c].astype("string")

# Optional: explode list columns if you want one row per function name
list_cols = [c for c in ["functions_standalone", "functions_class"] if c in df.columns]
if list_cols:
    df = df.explode(list_cols, ignore_index=True)

df


In [ ]:
#filter out non string entries in 'code_tokens' column
df = df[df['code_tokens'].apply(lambda x: isinstance(x, str))]
df

In [ ]:
# keep only rows where verdict column is "Accepted" or "Runtime Error"
df = df[df['verdict'].isin(['Accepted', 'Runtime Error'])]
df

In [ ]:
df['code_length'] = df['code_tokens'].apply(lambda x: len(x.split()))
df

In [ ]:
df['code_length'].describe()

In [ ]:
df['code_length'].plot(kind='hist', bins=20, title='Distribution of code snippet lengths', range=(0, 500))

In [ ]:
#use regular expressions to filter out rows where code_tokens column contains code snippets with very long lists of assignments
pattern = r"\[(?:[^,\]]+,){40,}[^,\]]+\]"   # ≥40 comma-separated elements

df["has_long_list"] = df["code_tokens"].str.contains(pattern, regex=True)

df = df[~df["has_long_list"]]   # remove snippets with long lists
df


In [ ]:
df = df[(df['code_length'] > 100) & (df['code_length'] < 1_000)]
df

In [ ]:
df['verdict'].value_counts()

In [ ]:
df['code_length_qualitative'] = pd.cut(df['code_length'], bins=[100, 300, 600, 1_000],
                                       labels=['short', 'medium', 'long'])
df

In [ ]:
df['code_length_qualitative'].value_counts()

In [ ]:
# sample 50 Accepted and 50 Runtime Error from each code_length_qualitative category
sampled_dfs = []
for length_category in ['short', 'medium', 'long']:
    category_df = df[df['code_length_qualitative'] == length_category]
    sampled_df = (category_df.groupby('verdict', group_keys=False).sample(n=50, random_state=42))
    sampled_dfs.append(sampled_df)
df_sample = pd.concat(sampled_dfs, ignore_index=True)

df_sample[['code_length_qualitative','verdict']].value_counts().reset_index()

In [ ]:
df_sample.to_csv("./data/code_snippets_sample.csv")